# Notebook 05 — Train / Validation / Test Split Verification

This notebook verifies that the BDD100K dataset has the expected **70,000 train + 10,000 validation + 20,000 test** split.

It also verifies that:
- every image has a valid split;
- there is no image ID overlap between train, validation, and test;
- the CycleGAN source images come only from the training split;
- the validation and test sets are not modified by the synthetic-fog process;
- the planned final dataset will contain approximately **73,000 train + 10,000 validation + 20,000 test = 103,000 images** if all 3,000 synthetic fog images pass quality control.

**Important:** This notebook does not create synthetic images and does not randomly re-split BDD100K. The original BDD100K split is treated as fixed.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")

METADATA_DIR = PROJECT_ROOT / "datasets" / "processed" / "metadata"

INPUT_CSV = METADATA_DIR / "bdd100k_processed_metadata.csv"
CONDITION_CSV = METADATA_DIR / "condition_labels.csv"
CYCLEGAN_SOURCE_CSV = METADATA_DIR / "cyclegan_source_images.csv"

print("PROJECT ROOT :", PROJECT_ROOT)
print("METADATA DIR :", METADATA_DIR)
print("INPUT CSV    :", INPUT_CSV)
print("CONDITION CSV:", CONDITION_CSV)
print("CYCLEGAN CSV :", CYCLEGAN_SOURCE_CSV)

PROJECT ROOT : C:\Users\ankit\Desktop\Adaptive_Road_Vision
METADATA DIR : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata
INPUT CSV    : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\bdd100k_processed_metadata.csv
CONDITION CSV: C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\condition_labels.csv
CYCLEGAN CSV : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\cyclegan_source_images.csv


In [2]:
# ============================================================
# CHECK REQUIRED FILES
# ============================================================

required_files = {
    "BDD100K processed metadata": INPUT_CSV,
    "Condition labels": CONDITION_CSV,
    "CycleGAN source metadata": CYCLEGAN_SOURCE_CSV,
}

missing = []

for name, path in required_files.items():
    exists = path.exists()
    print(f"{name:<35} : {exists}")
    if not exists:
        missing.append((name, path))

if missing:
    message = "\n".join([f"- {name}: {path}" for name, path in missing])
    raise FileNotFoundError(
        "The following required files are missing:\n" + message +
        "\n\nRun Notebook 03 and Notebook 04 successfully before continuing."
    )

print("\nAll required files are available.")

BDD100K processed metadata          : True
Condition labels                    : True
CycleGAN source metadata            : True

All required files are available.


In [3]:
# ============================================================
# LOAD METADATA
# ============================================================

df = pd.read_csv(INPUT_CSV)
condition_df = pd.read_csv(CONDITION_CSV)
cyclegan_df = pd.read_csv(CYCLEGAN_SOURCE_CSV)

print("Metadata loaded successfully.")
print("Metadata rows   :", len(df))
print("Metadata columns:", len(df.columns))

print("\nCondition labels rows:", len(condition_df))
print("CycleGAN source rows  :", len(cyclegan_df))

Metadata loaded successfully.
Metadata rows   : 100000
Metadata columns: 12

Condition labels rows: 100000
CycleGAN source rows  : 3000


In [4]:
# ============================================================
# BASIC METADATA VALIDATION
# ============================================================

required_columns = [
    "split",
    "image_name",
    "image_id",
    "image_path",
    "project_condition",
]

missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

print("Required columns found.")

# Normalize split values for validation only.
df["split"] = df["split"].astype(str).str.strip().str.lower()
df["image_id"] = df["image_id"].astype(str).str.strip().str.lower()

print("\nSplit values:")
print(df["split"].value_counts(dropna=False))

Required columns found.

Split values:
split
train    70000
test     20000
val      10000
Name: count, dtype: int64


## Expected BDD100K split

| Split | Expected images |
|---|---:|
| Train | 70,000 |
| Validation | 10,000 |
| Test | 20,000 |
| **Total** | **100,000** |

The notebook checks these values instead of changing the split.

In [5]:
# ============================================================
# VERIFY 70K / 10K / 20K SPLIT
# ============================================================

EXPECTED_SPLITS = {
    "train": 70_000,
    "val": 10_000,
    "test": 20_000,
}

split_counts = df["split"].value_counts().to_dict()

print("SPLIT VERIFICATION")
print("=" * 60)

for split_name, expected in EXPECTED_SPLITS.items():
    actual = int(split_counts.get(split_name, 0))
    status = "PASS" if actual == expected else "CHECK"
    print(f"{split_name.upper():<8} Expected: {expected:>7,} | Actual: {actual:>7,} | {status}")

print(f"{'TOTAL':<8} Expected: {100_000:>7,} | Actual: {len(df):>7,}")

if len(df) != 100_000:
    raise AssertionError(f"Expected 100,000 metadata records, found {len(df)}.")

for split_name, expected in EXPECTED_SPLITS.items():
    actual = int(split_counts.get(split_name, 0))
    if actual != expected:
        raise AssertionError(
            f"Split mismatch for {split_name}: expected {expected}, found {actual}."
        )

print("\nSUCCESS: 70K / 10K / 20K split is verified.")

SPLIT VERIFICATION
TRAIN    Expected:  70,000 | Actual:  70,000 | PASS
VAL      Expected:  10,000 | Actual:  10,000 | PASS
TEST     Expected:  20,000 | Actual:  20,000 | PASS
TOTAL    Expected: 100,000 | Actual: 100,000

SUCCESS: 70K / 10K / 20K split is verified.


In [6]:
# ============================================================
# CHECK DUPLICATE IMAGE IDs
# ============================================================

duplicate_mask = df["image_id"].duplicated(keep=False)
duplicate_count = int(duplicate_mask.sum())
        
print("Duplicate image ID rows:", duplicate_count)

if duplicate_count > 0:
    print("\nExample duplicate IDs:")
    print(df.loc[duplicate_mask, ["split", "image_name", "image_id"]].head(20).to_string(index=False))
    raise AssertionError("Duplicate image IDs detected in the 100K metadata.")

print("SUCCESS: Every image ID is unique.")

Duplicate image ID rows: 0
SUCCESS: Every image ID is unique.


In [7]:
# ============================================================
# VERIFY NO CROSS-SPLIT IMAGE OVERLAP
# ============================================================

train_ids = set(df.loc[df["split"] == "train", "image_id"])
val_ids = set(df.loc[df["split"] == "val", "image_id"])
test_ids = set(df.loc[df["split"] == "test", "image_id"])

train_val_overlap = train_ids & val_ids
train_test_overlap = train_ids & test_ids
val_test_overlap = val_ids & test_ids

print("TRAIN ∩ VAL  :", len(train_val_overlap))
print("TRAIN ∩ TEST :", len(train_test_overlap))
print("VAL ∩ TEST   :", len(val_test_overlap))

if train_val_overlap or train_test_overlap or val_test_overlap:
    raise AssertionError("Cross-split image overlap detected.")

print("\nSUCCESS: Train, validation, and test image IDs are disjoint.")

TRAIN ∩ VAL  : 0
TRAIN ∩ TEST : 0
VAL ∩ TEST   : 0

SUCCESS: Train, validation, and test image IDs are disjoint.


In [8]:
# ============================================================
# CONDITION DISTRIBUTION BY SPLIT
# ============================================================

condition_table = pd.crosstab(
    df["split"],
    df["project_condition"],
    margins=True
)

print("CONDITION × SPLIT")
print("=" * 60)
display(condition_table)

print("\nCondition distribution by split (%):")
condition_percent = pd.crosstab(
    df["split"],
    df["project_condition"],
    normalize="index"
) * 100

display(condition_percent.round(2))

CONDITION × SPLIT
project_condition    Day  Fog  Night  Rain  Unknown     All
split                                                      
test                9817   38   8798  1306       41   20000
train              34220  130  30441  5083      126   70000
val                 4857   13   4358   738       34   10000
All                48894  181  43597  7127      201  100000

Condition distribution by split (%):
project_condition    Day   Fog  Night  Rain  Unknown
split                                               
test               49.08  0.19  43.99  6.53     0.20
train              48.89  0.19  43.49  7.26     0.18
val                48.57  0.13  43.58  7.38     0.34


In [9]:
# ============================================================
# VERIFY CONDITION LABEL FILE HAS THE SAME IMAGE COVERAGE
# ============================================================

if "image_id" not in condition_df.columns:
    if "image_name" not in condition_df.columns:
        raise KeyError(
            "condition_labels.csv must contain either 'image_id' or 'image_name'."
        )

    condition_df["image_id"] = (
        condition_df["image_name"].astype(str).str.strip().str.lower()
    )

condition_ids = set(
    condition_df["image_id"].astype(str).str.strip().str.lower()
)

metadata_ids = set(df["image_id"])

missing_from_condition = metadata_ids - condition_ids
extra_in_condition = condition_ids - metadata_ids

print("Metadata IDs           :", len(metadata_ids))
print("Condition label IDs    :", len(condition_ids))
print("Missing condition IDs  :", len(missing_from_condition))
print("Extra condition IDs    :", len(extra_in_condition))

if missing_from_condition or extra_in_condition:
    raise AssertionError("Condition label coverage does not exactly match metadata.")

print("\nSUCCESS: Condition labels cover all 100,000 metadata records.")

Metadata IDs           : 100000
Condition label IDs    : 100000
Missing condition IDs  : 0
Extra condition IDs    : 0

SUCCESS: Condition labels cover all 100,000 metadata records.


In [10]:
# ============================================================
# VERIFY CYCLEGAN SOURCES
# ============================================================

print("CYCLEGAN SOURCE VERIFICATION")
print("=" * 60)

print("Rows:", len(cyclegan_df))

# ------------------------------------------------------------
# 1. CHECK AVAILABLE COLUMNS
# ------------------------------------------------------------

print("\nCycleGAN CSV columns:")
for col in cyclegan_df.columns:
    print(" -", col)


# ------------------------------------------------------------
# 2. GET CYCLEGAN IMAGE IDs
# ------------------------------------------------------------

if "image_id" in cyclegan_df.columns:

    # Preferred column
    cyclegan_df["image_id"] = (
        cyclegan_df["image_id"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

elif "image_name" in cyclegan_df.columns:

    # Use image_name when image_id is not present
    cyclegan_df["image_id"] = (
        cyclegan_df["image_name"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

else:

    raise KeyError(
        "\nThe CycleGAN source CSV must contain either:\n"
        "  - image_id\n"
        "  - image_name\n\n"
        f"Available columns are:\n{list(cyclegan_df.columns)}"
    )


# ------------------------------------------------------------
# 3. CHECK EXACTLY 3,000 SOURCE RECORDS
# ------------------------------------------------------------

if len(cyclegan_df) != 3_000:

    raise AssertionError(
        f"\nExpected exactly 3,000 CycleGAN source records, "
        f"but found {len(cyclegan_df)}."
    )


# ------------------------------------------------------------
# 4. CREATE UNIQUE CYCLEGAN IMAGE ID SET
# ------------------------------------------------------------

cyclegan_ids = set(
    cyclegan_df["image_id"]
)


print("\nUnique CycleGAN source IDs :", len(cyclegan_ids))
print("Total CycleGAN rows        :", len(cyclegan_df))


# ------------------------------------------------------------
# 5. CHECK DUPLICATES
# ------------------------------------------------------------

if len(cyclegan_ids) != len(cyclegan_df):

    duplicate_ids = (
        cyclegan_df[
            cyclegan_df["image_id"].duplicated(keep=False)
        ]["image_id"]
        .unique()
        .tolist()
    )

    print("\nDuplicate image IDs detected:")
    print(duplicate_ids[:20])

    raise AssertionError(
        "Duplicate image IDs found in CycleGAN source metadata."
    )

print("Duplicate CycleGAN IDs    : 0")


# ------------------------------------------------------------
# 6. CHECK AGAINST BDD100K METADATA
# ------------------------------------------------------------

source_in_metadata = cyclegan_ids & metadata_ids
source_not_in_metadata = cyclegan_ids - metadata_ids

print("\nSources found in metadata :", len(source_in_metadata))
print("Sources NOT in metadata   :", len(source_not_in_metadata))


if source_not_in_metadata:

    print("\nExample missing IDs:")
    print(
        list(sorted(source_not_in_metadata))[:20]
    )

    raise AssertionError(
        "Some CycleGAN source images are not present "
        "in the BDD100K processed metadata."
    )


print("\nSUCCESS: All CycleGAN source images exist in BDD100K metadata.")


# ------------------------------------------------------------
# 7. JOIN WITH AUTHORITATIVE BDD100K METADATA
# ------------------------------------------------------------

source_check = df[
    df["image_id"].isin(cyclegan_ids)
].copy()


print("\nMatched source records in BDD100K:", len(source_check))


if len(source_check) != 3_000:

    raise AssertionError(
        f"Expected 3,000 matched CycleGAN records, "
        f"found {len(source_check)}."
    )


# ------------------------------------------------------------
# 8. VERIFY ALL SOURCES ARE FROM TRAIN
# ------------------------------------------------------------

print("\nCycleGAN source split:")
print(
    source_check["split"]
    .value_counts()
)


non_train_sources = source_check[
    source_check["split"] != "train"
]


if len(non_train_sources) > 0:

    print("\nNon-training CycleGAN sources detected:")
    print(
        non_train_sources[
            ["image_id", "split"]
        ].head(20).to_string(index=False)
    )

    raise AssertionError(
        "DATA LEAKAGE: CycleGAN sources must come ONLY "
        "from the training split."
    )


print(
    "\nSUCCESS: All 3,000 CycleGAN sources are from TRAIN."
)


# ------------------------------------------------------------
# 9. VERIFY CONDITION COMPOSITION
# ------------------------------------------------------------

print("\nCycleGAN source conditions:")
print(
    source_check["project_condition"]
    .value_counts()
)


day_count = int(
    (source_check["project_condition"] == "Day").sum()
)

night_count = int(
    (source_check["project_condition"] == "Night").sum()
)


print("\nCondition verification:")
print("Day sources   :", day_count)
print("Night sources :", night_count)
print("Total         :", day_count + night_count)


# ------------------------------------------------------------
# 10. VERIFY EXPECTED 1,500 + 1,500
# ------------------------------------------------------------

if day_count != 1_500:

    raise AssertionError(
        f"Expected 1,500 Day sources, found {day_count}."
    )


if night_count != 1_500:

    raise AssertionError(
        f"Expected 1,500 Night sources, found {night_count}."
    )


if day_count + night_count != 3_000:

    raise AssertionError(
        "CycleGAN source composition does not total 3,000."
    )


# ------------------------------------------------------------
# 11. FINAL SUCCESS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CYCLEGAN SOURCE VERIFICATION PASSED")
print("=" * 60)

print("Total sources       : 3,000")
print("Day sources         : 1,500")
print("Night sources       : 1,500")
print("Training sources    : 3,000")
print("Validation sources  : 0")
print("Test sources        : 0")
print("Duplicate IDs       : 0")
print("Missing metadata    : 0")

print(
    "\nSUCCESS: CycleGAN sources are correctly prepared "
    "for the next stage."
)

CYCLEGAN SOURCE VERIFICATION
Rows: 3000

CycleGAN CSV columns:
 - split
 - image_name
 - weather_label
 - time_label
 - condition
 - cyclegan_source_domain
 - synthetic_target_domain
 - annotation_file

Unique CycleGAN source IDs : 3000
Total CycleGAN rows        : 3000
Duplicate CycleGAN IDs    : 0

Sources found in metadata : 3000
Sources NOT in metadata   : 0

SUCCESS: All CycleGAN source images exist in BDD100K metadata.

Matched source records in BDD100K: 3000

CycleGAN source split:
split
train    3000
Name: count, dtype: int64

SUCCESS: All 3,000 CycleGAN sources are from TRAIN.

CycleGAN source conditions:
project_condition
Day      1500
Night    1500
Name: count, dtype: int64

Condition verification:
Day sources   : 1500
Night sources : 1500
Total         : 3000

CYCLEGAN SOURCE VERIFICATION PASSED
Total sources       : 3,000
Day sources         : 1,500
Night sources       : 1,500
Training sources    : 3,000
Validation sources  : 0
Test sources        : 0
Duplicate IDs       :

In [11]:
# ============================================================
# VERIFY CYCLEGAN SOURCE CONDITION COMPOSITION
# ============================================================

source_conditions = source_check["project_condition"].value_counts().to_dict()

day_count = int(source_conditions.get("Day", 0))
night_count = int(source_conditions.get("Night", 0))

print(f"Day source images  : {day_count}")
print(f"Night source images: {night_count}")
print(f"Total sources      : {day_count + night_count}")

if day_count != 1_500:
    raise AssertionError(f"Expected 1,500 Day sources, found {day_count}.")

if night_count != 1_500:
    raise AssertionError(f"Expected 1,500 Night sources, found {night_count}.")

if day_count + night_count != 3_000:
    raise AssertionError("CycleGAN source composition does not total 3,000.")

print("\nSUCCESS: CycleGAN sources = 1,500 Day + 1,500 Night.")

Day source images  : 1500
Night source images: 1500
Total sources      : 3000

SUCCESS: CycleGAN sources = 1,500 Day + 1,500 Night.


In [12]:
# ============================================================
# LEAKAGE CHECK — CYCLEGAN SOURCES MUST NOT BE VAL/TEST
# ============================================================

val_test_ids = val_ids | test_ids
leakage_ids = cyclegan_ids & val_test_ids

print("CycleGAN source IDs in VAL/TEST:", len(leakage_ids))

if leakage_ids:
    print("Examples:")
    print(list(sorted(leakage_ids))[:20])
    raise AssertionError(
        "DATA LEAKAGE DETECTED: CycleGAN source images overlap validation/test."
    )

print("SUCCESS: No CycleGAN source leakage into validation/test.")

CycleGAN source IDs in VAL/TEST: 0
SUCCESS: No CycleGAN source leakage into validation/test.


## Planned final dataset

The original BDD100K split remains fixed:

- Train = 70,000
- Validation = 10,000
- Test = 20,000

CycleGAN is planned to generate approximately 3,000 synthetic fog images **from training images only**:

- 1,500 Clear+Day → synthetic Day+Fog
- 1,500 Clear+Night → synthetic Night+Fog

If all 3,000 synthetic images pass quality control, the final dataset will be:

**73,000 train + 10,000 validation + 20,000 test = 103,000 images.**

The actual final count must be calculated after synthetic-fog quality control; rejected synthetic images are not included.

In [13]:
# ============================================================
# FINAL SPLIT PLAN SUMMARY
# ============================================================

ORIGINAL_TRAIN = int((df["split"] == "train").sum())
ORIGINAL_VAL = int((df["split"] == "val").sum())
ORIGINAL_TEST = int((df["split"] == "test").sum())

PLANNED_SYNTHETIC = 3_000

PLANNED_FINAL_TRAIN = ORIGINAL_TRAIN + PLANNED_SYNTHETIC
PLANNED_FINAL_VAL = ORIGINAL_VAL
PLANNED_FINAL_TEST = ORIGINAL_TEST
PLANNED_FINAL_TOTAL = (
    PLANNED_FINAL_TRAIN + PLANNED_FINAL_VAL + PLANNED_FINAL_TEST
)

summary = pd.DataFrame({
    "split": ["train", "val", "test", "total"],
    "original_images": [
        ORIGINAL_TRAIN,
        ORIGINAL_VAL,
        ORIGINAL_TEST,
        len(df),
    ],
    "planned_synthetic_added": [
        PLANNED_SYNTHETIC,
        0,
        0,
        PLANNED_SYNTHETIC,
    ],
    "planned_final_images": [
        PLANNED_FINAL_TRAIN,
        PLANNED_FINAL_VAL,
        PLANNED_FINAL_TEST,
        PLANNED_FINAL_TOTAL,
    ],
})

display(summary)

print("Validation/test remain untouched by the planned synthetic augmentation.")

   split  original_images  planned_synthetic_added  planned_final_images
0  train            70000                     3000                 73000
1    val            10000                        0                 10000
2   test            20000                        0                 20000
3  total           100000                     3000                103000
Validation/test remain untouched by the planned synthetic augmentation.


In [14]:
# ============================================================
# SAVE SPLIT VERIFICATION REPORT
# ============================================================

REPORT_PATH = METADATA_DIR / "split_verification_report.csv"

summary.to_csv(REPORT_PATH, index=False)

print("Saved split verification report:")
print(REPORT_PATH)
print("File exists:", REPORT_PATH.exists())

Saved split verification report:
C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\split_verification_report.csv
File exists: True


In [15]:
# ============================================================
# FINAL ASSERTIONS
# ============================================================

assert len(df) == 100_000
assert ORIGINAL_TRAIN == 70_000
assert ORIGINAL_VAL == 10_000
assert ORIGINAL_TEST == 20_000
assert not (train_ids & val_ids)
assert not (train_ids & test_ids)
assert not (val_ids & test_ids)
assert len(cyclegan_ids) == 3_000
assert not (cyclegan_ids & val_test_ids)
assert day_count == 1_500
assert night_count == 1_500

print("=" * 70)
print("NOTEBOOK 05 — ALL CHECKS PASSED")
print("=" * 70)
        
print("Original dataset : 100,000")
print("Train           : 70,000")
print("Validation      : 10,000")
print("Test            : 20,000")
print("CycleGAN sources: 3,000")
print("  Day           : 1,500")
print("  Night         : 1,500")
print("Synthetic data  : TRAIN ONLY")
print("Leakage         : 0")
print("Planned final   : 103,000 if all synthetic images pass QC")

print("\nREADY FOR NOTEBOOK 06 — CYCLEGAN DATASET PREPARATION")

NOTEBOOK 05 — ALL CHECKS PASSED
Original dataset : 100,000
Train           : 70,000
Validation      : 10,000
Test            : 20,000
CycleGAN sources: 3,000
  Day           : 1,500
  Night         : 1,500
Synthetic data  : TRAIN ONLY
Leakage         : 0
Planned final   : 103,000 if all synthetic images pass QC

READY FOR NOTEBOOK 06 — CYCLEGAN DATASET PREPARATION


# Completion

After all cells pass, proceed to:

**`06_cyclegan_dataset_preparation.ipynb`**

That notebook will prepare the actual CycleGAN folders from the 3,000 selected training images. It should not modify the original BDD100K images.